<a href="https://colab.research.google.com/github/arbin34/heruk/blob/main/videoanlysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary packages (Ultralytics YOLOv8, OpenCV, DeepSORT)
!pip install ultralytics opencv-python-headless deep-sort-realtime

import cv2
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
from collections import defaultdict


In [ ]:
from google.colab import files
import requests

print("Upload your tennis video file (mp4 or webm). If no file is uploaded, a sample video will be downloaded automatically.")
uploaded = files.upload()

if not uploaded:
    # Download a short Creative Commons tennis clip from Wikimedia Commons
    sample_url = "https://upload.wikimedia.org/wikipedia/commons/5/55/2018_Davis_Cup_Americas_Zone_-_Uruguay_vs_Mexico_-_01.webm"
    res = requests.get(sample_url)
    with open("sample_tennis.webm", "wb") as f:
        f.write(res.content)
    video_path = "sample_tennis.webm"
else:
    video_path = list(uploaded.keys())[0]

print(f"Using video: {video_path}")


In [3]:
# Load YOLOv8 model (pre-trained on COCO, default person detection is class 0)
model = YOLO("yolov8n.pt")

# Initialize DeepSORT tracker with a specified max age (frames to keep lost tracks)
tracker = DeepSort(max_age=30)

# Set detection confidence threshold
CONFIDENCE_THRESHOLD = 0.3  # only consider detections above this confidence


100%|██████████| 6.25M/6.25M [00:00<00:00, 82.4MB/s]


In [5]:
detect_court = False  # Set to True to detect and draw court lines

if detect_court:
    print("Court line detection enabled (lines will be overlaid).")
else:
    print("Court line detection disabled.")


Court line detection disabled.


In [ ]:
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise IOError(f"Cannot open video: {video_path}")

# Prepare video writer for output
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter("output.mp4", fourcc, fps, (width, height))

# Dictionaries to hold track history (for trails) and colors
track_history = defaultdict(list)
colors = {}

frame_idx = 0
lines = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detect court lines on first frame if enabled
    if detect_court and frame_idx == 0:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150)
        lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100,
                                minLineLength=100, maxLineGap=10)

    # Run YOLOv8 detection on the frame
    results = model(frame)[0]
    detections = []
    for data in results.boxes.data.tolist():
        x1, y1, x2, y2, conf, cls_id = data
        if conf < CONFIDENCE_THRESHOLD or int(cls_id) != 0:
            continue
        # Format: [x, y, width, height], confidence, class
        detections.append([[int(x1), int(y1), int(x2-x1), int(y2-y1)], float(conf), int(cls_id)])

    # Update DeepSORT tracker with new detections
    tracks = tracker.update_tracks(detections, frame=frame)

    # Draw tracking results
    for track in tracks:
        if not track.is_confirmed():
            continue
        track_id = track.track_id
        l, t, r, b = track.to_ltrb()
        xmin, ymin, xmax, ymax = int(l), int(t), int(r), int(b)

        # Assign a consistent color to each track ID
        if track_id not in colors:
            colors[track_id] = tuple(np.random.randint(0, 255, 3).tolist())
        color = colors[track_id]

        # Draw bounding box and label (ID)
        cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), color, 2)
        cv2.putText(frame, f"ID {track_id}", (xmin, ymin-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)

        # Update and draw movement trail for this track
        center = (int((xmin+xmax)/2), int((ymin+ymax)/2))
        track_history[track_id].append(center)
        # Keep only recent points for the trail (e.g., last 30 points)
        if len(track_history[track_id]) > 30:
            track_history[track_id].pop(0)
        for i in range(1, len(track_history[track_id])):
            cv2.line(frame, track_history[track_id][i-1], track_history[track_id][i], color, 2)

    # Overlay detected court lines if enabled
    if detect_court and lines is not None:
        for x1, y1, x2, y2 in lines[:,0]:
            cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print("Processing complete. Annotated video saved as output.mp4.")


In [7]:
from google.colab import files
files.download("output.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>